# HFusionHub RAG 评估 Notebook

对 HFusionHub 的 RAG 检索引擎做**离线批量评估**：加载数据集 → 调用内置评估端点
`POST /api/rag/evaluate`（与「回答效果」页同一条生产链路）→ 展示 Precision/Recall/MRR 指标与失败用例。

**前置条件**:
- Python AI 服务运行在 `localhost:9000`
- 环境变量 `PYTHON_AI_INTERNAL_TOKEN` 与服务端一致（生产由 compose 强制注入）
- 环境变量 `EVAL_TENANT_ID` 为目标租户（默认 1）
- 至少有一个已解析的知识库；数据集 JSONL 每行包含
  `query` 与 `expected_document_ids`（文档 ID 列表，可在文档页/详情接口查看）

> 与旧版差异：不再手写指标计算（旧实现从 trace 提取 `document_title`，该字段
> 已不存在导致指标恒为 0），统一走内置评估器 —— 指标口径与「回答效果」页一致，
> 且每次评估会持久化为 evaluation run（可在 `/rag` 页查看历史）。


In [ ]:
import json
import os
import urllib.request
from pathlib import Path

PYTHON_AI_URL = os.getenv("PYTHON_AI_URL", "http://localhost:9000")
INTERNAL_TOKEN = os.getenv("PYTHON_AI_INTERNAL_TOKEN", "")
TENANT_ID = os.getenv("EVAL_TENANT_ID", "1")
DATASET_PATH = Path(os.getenv("EVAL_DATASET", "../eval_example.jsonl"))
KNOWLEDGE_BASE_ID = int(os.getenv("EVAL_KB_ID", "132"))  # 替换为你的知识库 ID
TOP_K = 5

HEADERS = {
    "X-Internal-Token": INTERNAL_TOKEN,
    "X-Tenant-Id": TENANT_ID,
    "Content-Type": "application/json",
}
assert INTERNAL_TOKEN, "请设置环境变量 PYTHON_AI_INTERNAL_TOKEN（与 python-ai 服务一致）"
print(f"Python AI: {PYTHON_AI_URL} | KB: {KNOWLEDGE_BASE_ID} | tenant: {TENANT_ID}")
print(f"Dataset: {DATASET_PATH.resolve()}")

In [ ]:
NL = "\n"

def load_dataset(path: Path) -> list[dict]:
    """加载 JSONL 数据集。兼容旧字段 expected_documents（文件名示意），
    但评估端点期望 expected_document_ids（真实文档 ID），建议数据集使用后者。"""
    samples = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                samples.append(json.loads(line))
    return samples


def build_cases(samples: list[dict]) -> list[dict]:
    cases = []
    for i, s in enumerate(samples):
        ids = s.get("expected_document_ids") or s.get("expected_documents") or []
        if not ids:
            print(f"  [skip] 样本 {i} 无期望文档: {s['query'][:50]}")
            continue
        cases.append({
            "case_id": s.get("case_id") or f"case-{i + 1}",
            "query": s["query"],
            "expected_document_ids": [str(x) for x in ids],
        })
    return cases


def run_evaluation(cases: list[dict], kb_id: int, top_k: int = TOP_K) -> dict:
    body = {"knowledge_base_id": kb_id, "top_k": top_k,
            "label": "notebook", "cases": cases}
    req = urllib.request.Request(
        f"{PYTHON_AI_URL}/api/rag/evaluate",
        data=json.dumps(body, ensure_ascii=False).encode("utf-8"),
        method="POST", headers=HEADERS,
    )
    with urllib.request.urlopen(req, timeout=120) as resp:
        return json.loads(resp.read())


def show_failures(result: dict, top_n: int = 5):
    """展示未命中用例（reciprocal_rank = 0 表示第一页没有任何期望文档）。"""
    failures = [c for c in result.get("cases", []) if not c.get("reciprocal_rank")]
    print(f"{NL}### 未命中用例: {len(failures)}/{len(result.get('cases', []))}")
    for c in failures[:top_n]:
        print(f"  - {c['query'][:70]}")
        print(f"    期望: {c['expected_document_ids']}  实际: {c['retrieved_document_ids'][:5]}")

### 运行评估

> 示例数据集 `eval_example.jsonl` 的期望文档是文件名示意（`rag_intro.pdf` 等），
> 对真实知识库评估时请构造包含真实 `expected_document_ids` 的数据集。

In [ ]:
samples = load_dataset(DATASET_PATH)
print(f"加载 {len(samples)} 条样本")
cases = build_cases(samples)
print(f"有效用例 {len(cases)} 条，开始评估...")

result = run_evaluation(cases, KNOWLEDGE_BASE_ID)

summary = result.get("summary", {})
print(f"{NL}### 指标汇总")
for k, v in summary.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

run = result.get("run", {})
if run:
    print(f"{NL}已持久化为 evaluation run: {run.get('run_id')}（可在「回答效果」页查看历史）")

show_failures(result)

## 指标说明与优化方向

| 指标 | 说明 | 参考目标 |
|------|------|---------|
| precision_at_k | Top-K 中期望文档的占比 | ≥ 0.60 |
| recall_at_k | 期望文档被 Top-K 召回的占比 | ≥ 0.90 |
| mean_reciprocal_rank | 第一个期望文档的平均排名倒数 | ≥ 0.70 |
| graph_hit_rate / multimodal_hit_rate | 历史通道指标（GraphRAG 已移除，恒为 0；多模态按需） | — |

**常见优化方向**:
- Recall 低 → 确认 `RAG_HYBRID_ENABLED=true`（向量 + 关键词混合，默认开）
- 排名低（MRR 低）→ `RAG_RERANKER_MODE=lexical`，语义要求高可试 `cross_encoder`
  （离线基准见 `scripts/eval_reranker.py`）
- 单条深查：`POST /api/rag/debug/search` 返回完整 trace（每通道候选与存活原因）
- 指标历史：`GET /api/rag/evaluation-runs?knowledge_base_id=<id>`
